---
title: "بخش دوم: داده‌ها و عملگرها"
authors:
  - name: دکتر محمود امین طوسی
    affiliations:
      - institution: دانشگاه فردوسی مشهد
        department: گروه علوم کامپیوتر
        country: ایران
    email: m.amintoosi@gmail.com
    github: mamintoosi
    url: https://mamintoosi.github.io/
    roles: [supervision]
  - name: ملیکا محمدبیگی
    affiliations:
      - institution: دانشگاه فردوسی مشهد
        department: گروه علوم کامپیوتر
    github: mbeygi569-ux
  - name: درسا عناصری
    affiliations:
      - institution: دانشگاه فردوسی مشهد
        department: گروه علوم کامپیوتر
    github: Anasori-Dorsa
  - name: زینب قاراچشمه
    affiliations:
      - institution: دانشگاه فردوسی مشهد
        department: گروه علوم کامپیوتر
    github: eowyn2006
---

# بخش دوم: داده‌ها و عملگرها

<div dir="rtl" style="text-align:right">

در سایت کتاب، نمونه‌های C همراه با خروجی‌های ذخیره‌شده نمایش داده می‌شوند. برای اجرای مثال‌ها، نوت‌بوک را از منوی **Downloads** بگیرید و در Google Colab یا Jupyter باز کنید.

**اجرای مثال‌ها در نوت‌بوک (با `gcc`):**
- ابتدا سلول «آماده‌سازی» را اجرا کنید.
- هر سلول کدی که با `%%c` شروع می‌شود با `gcc` کامپایل و اجرا می‌شود. اگر برنامه ورودی می‌خواهد، آن را جلوی `%%c` بنویسید؛ مثلاً `%%c 25\n30` یعنی دو خط ورودی «۲۵» و «۳۰».
- تکه‌کدهایی که `main` ندارند خودکار داخل `main` قرار می‌گیرند. چند نمونه‌ی «غلط‌نمایی» عمداً فقط نمایشی‌اند و اجرا نمی‌شوند.

</div>

In [1]:
# آماده‌سازی: ثبت دستور %%c برای کامپایل و اجرای کد C
import re, subprocess, tempfile, os
from IPython.core.magic import register_cell_magic

_HEADERS = ("#include <stdio.h>\n#include <stdlib.h>\n#include <string.h>\n#include <limits.h>\n"
            "#include <float.h>\n#include <stdint.h>\n#include <stdbool.h>\n#include <math.h>\n#include <stddef.h>\n")

def run_c(code, stdin_text=""):
    """کد C را کامپایل و اجرا می‌کند. اگر main نداشته باشد، تکه‌کد را داخل main می‌گذارد."""
    if not re.search(r'\bmain\s*\(', code):
        lines = code.split("\n")
        top = [l for l in lines if l.lstrip().startswith("#")]
        body = [l for l in lines if not l.lstrip().startswith("#")]
        code = _HEADERS + "\n".join(top) + "\nint main(void) {\n" + "\n".join(body) + "\nreturn 0;\n}\n"
    with tempfile.TemporaryDirectory() as d:
        src, exe = os.path.join(d, "prog.c"), os.path.join(d, "prog")
        open(src, "w", encoding="utf-8").write(code)
        # نمونه‌های کوتاه کتاب متغیرهایی را فقط برای نمایش تعریف می‌کنند؛
        # فقط هشدارهای «متغیر بلااستفاده» بی‌صدا می‌شوند و بقیه‌ی هشدارها فعال می‌مانند.
        cp = subprocess.run(["gcc", "-Wall", "-Wextra", "-Wno-unused-variable",
                             "-Wno-unused-but-set-variable", "-std=c11", src, "-o", exe, "-lm"],
                            capture_output=True, text=True)
        out = ""
        if cp.stderr.strip():
            out += "--- پیام‌های کامپایلر ---\n" + cp.stderr.replace(src, "prog.c") + "\n"
        if cp.returncode != 0:
            return out + "(کامپایل ناموفق بود)"
        try:
            rp = subprocess.run([exe], input=stdin_text, capture_output=True, text=True, timeout=5)
            if rp.stdout or rp.stderr:
                out += "--- خروجی برنامه ---\n" + rp.stdout + rp.stderr
            if rp.returncode != 0:
                out += f"\n(کد خروج: {rp.returncode})"
        except subprocess.TimeoutExpired:
            out += "(اجرا بعد از ۵ ثانیه متوقف شد)"
        return out

@register_cell_magic
def c(line, cell):
    """%%c [ورودی دلخواه برنامه؛ با \\n برای خط جدید]"""
    output = run_c(cell, line.encode().decode("unicode_escape"))
    if output:
        print(output)

در برنامه‌نویسی، قبل از آن‌که بتوانیم مسئله‌ای را با شرط‌ها، حلقه‌ها یا توابع حل کنیم، باید بدانیم داده‌ها را چطور نمایش می‌دهیم و با آن‌ها چه عملیاتی انجام می‌دهیم. فرض کنید می‌خواهیم سن چند نفر را نگه داریم، میانگین نمره‌ای را حساب کنیم، از کاربر یک عدد بگیریم یا بررسی کنیم که یک مقدار زوج است یا فرد. برای انجام همین کارهای ساده هم باید بدانیم هر مقدار چه نوعی دارد، در حافظه چگونه نمایش داده می‌شود و عملگرهای زبان C دقیقاً چه نتیجه‌ای تولید می‌کنند.

این بخش از همین نقطه شروع می‌کند. ابتدا سراغ **متغیرها و انواع داده** می‌رویم تا ببینیم داده‌ها چگونه در حافظه نگه‌داری می‌شوند، نام‌گذاری و مقداردهی آن‌ها چه قواعدی دارد و تبدیل نوع یا سرریز چه اثری بر نتیجه می‌گذارد. بعد یاد می‌گیریم با **`printf` و `scanf`** اطلاعات را از برنامه خارج کنیم یا از کاربر بگیریم. در ادامه، عملگرهای محاسباتی، رابطه‌ای، منطقی، انتسابی و بیتی را بررسی می‌کنیم و در پایان به این می‌رسیم که ترکیب چند عملگر در یک عبارت چگونه تفسیر می‌شود.

در طول فصل‌ها، از مثال‌هایی مثل محاسبه‌ی سن و میانگین، دریافت نمره، بررسی PIN، تبدیل واحدها و کار با بیت‌ها استفاده می‌کنیم. هدف این است که فقط شکل نوشتن دستورها را به خاطر نسپاریم؛ بلکه بتوانیم مسیر رسیدن از داده‌ی ورودی به نتیجه‌ی نهایی را تحلیل کنیم و علت خطاهای رایج را تشخیص دهیم.

---

## فصل ۱ : متغیرها و انواع داده

### 1.1 چرا به متغیر و نوع داده نیاز داریم؟

فرض کنید می‌خواهیم برنامه‌ای بنویسیم که سن یک دانشجو، نمره‌ی او و معدلش را نگه دارد. برنامه باید بتواند این مقدارها را در محاسبات به کار ببرد، آن‌ها را تغییر دهد و در زمان مناسب نمایش دهد. اگر داده‌ها را فقط به‌صورت عددهای پراکنده در کد بنویسیم، راه منظمی برای نام‌گذاری، نگه‌داری و استفاده‌ی دوباره از آن‌ها نخواهیم داشت. برای حل این مسئله به مفهومی نیاز داریم که **متغیر (Variable)** نام دارد.

متغیر را می‌توان مثل یک خانه‌ی نام‌گذاری‌شده در حافظه تصور کرد که در هر لحظه مقداری را نگه می‌دارد. برای مثال:



In [2]:
%%c
int age = 20;



در این دستور، `age` نام متغیر، `int` نوع داده و `20` مقدار اولیه‌ی آن است. نوع داده مشخص می‌کند مقدار چگونه تفسیر شود و چه بازه‌ای از مقدارها را می‌تواند نگه دارد؛ بنابراین انتخاب نوع مناسب، بخشی از حل مسئله است، نه صرفاً نوشتن دستور به شکل درست.

> **نگاهی به یک دانشمند تأثیرگذار — دنیس ریچی (Dennis Ritchie):**
> دنیس ریچی، دانشمند علوم کامپیوتر آمریکایی، زبان C را در آزمایشگاه Bell در اوایل دهه‌ی ۱۹۷۰ طراحی و توسعه داد. C ابتدا برای پیاده‌سازی بخش‌هایی از سیستم‌عامل Unix به کار رفت و به‌دلیل ترکیب نزدیکی به سخت‌افزار با قابلیت حمل و ساختار نسبتاً ساده، به یکی از زبان‌های اثرگذار در تاریخ برنامه‌نویسی تبدیل شد. بسیاری از مفاهیمی که در این بخش می‌آموزیم، در قالب زبان C و کتابخانه‌ی استاندارد آن به کار گرفته می‌شوند.

#### سه مفهوم که نباید با هم قاطی شوند: نام، نوع و مقدار

در سطح مقدماتی می‌گوییم «متغیر مثل جعبه است». این تشبیه مفید است، اما از همین ابتدا بهتر است سه چیز را جدا ببینیم:

1. **نام متغیر** (`age`) چیزی است که برنامه‌نویس در کد می‌نویسد.
2. **نوع متغیر** (`int`) به کامپایلر می‌گوید این مقدار چگونه تفسیر شود و چه عملیات‌هایی روی آن معنادار است.
3. **مقدار** (`20`) محتوای فعلی آن متغیر است؛ در زمان اجرا این مقدار در بخشی از حافظه نگه‌داری می‌شود.

> **پنجره‌ی بیشتر:** چند فصل بعد، با عملگر `&` می‌توان آدرس یک متغیر را دید و با «اشاره‌گر» مستقیماً درباره‌ی آدرس‌ها صحبت کرد. در این بخش فقط کافی است بدانیم متغیر علاوه بر نام و مقدار، در زمان اجرا یک محل ذخیره هم دارد.

#### Declaration، Initialization و Assignment

این سه واژه شبیه هم‌اند اما یکی نیستند:



In [3]:
%%c
int score;       // Declaration: معرفی متغیر (مقدار اولیه نامشخص است!)
int age = 18;    // Declaration + Initialization: تعریف همراه با مقداردهی اولیه
score = 20;      // Assignment: تغییر یا تعیین مقدار پس از تعریف



**نکته‌ی امتحانی و رایج:** متغیری که فقط Declaration شده باشد (`int score;`)، مقدارش صفر نیست؛ بلکه مقدار زباله (Garbage Value) دارد و خواندن آن پیش از مقداردهی، خطای جدی و رفتار تعریف‌نشده ایجاد می‌کند.

### 1.2 انواع داده پایه

زبان C چند نوع داده‌ی پایه دارد که هرکدام برای ذخیره‌ی نوع خاصی از اطلاعات مناسب‌اند:

| نوع داده    | کاربرد                                          | اندازه‌ی معمول (بایت) | مثال                           |
|-------------|-------------------------------------------------|----------------------|--------------------------------|
| `int`       | اعداد صحیح                                      | ۴                    | `int x = 10;`                  |
| `long`      | اعداد صحیح بزرگ‌تر                               | ۴ یا ۸               | `long x = 100000L;`            |
| `long long` | اعداد صحیح خیلی بزرگ                            | ۸                    | `long long x = 10000000000LL;` |
| `float`     | اعداد اعشاری (دقت کم)                           | ۴                    | `float pi = 3.14f;`            |
| `double`    | اعداد اعشاری (دقت بالا)                         | ۸                    | `double pi = 3.14159265;`      |
| `char`      | یک کاراکتر (و عدد صحیح کوچک ۱ بایتی)            | ۱                    | `char grade = 'A';`            |
| `bool`      | مقدار درست/غلط (نیاز به `#include <stdbool.h>`) | ۱                    | `bool isValid = true;`         |

**نمایش بصری اشغال حافظه:** هر خانه‌ی حافظه یک بایت و هر بایت شامل ۸ بیت است. اندازه‌های معمول روی بسیاری از سیستم‌های امروزی عبارت‌اند از: `char` و `bool` برابر ۱ بایت، `int` و `float` برابر ۴ بایت و `double` برابر ۸ بایت. این اندازه‌ها همگی تضمین‌شده نیستند و برای بررسی اندازه‌ی واقعی روی سیستم خود باید از عملگر `sizeof` استفاده کرد:



In [4]:
%%c
printf("%zu\n", sizeof(int)); // معمولاً چاپ می‌کند: 4


--- خروجی برنامه ---
4




#### تفاوت `float` و `double`

هر دو برای اعداد اعشاری هستند، اما `double` حدود ۱۵ تا ۱۶ رقم معنی‌دار دقت دارد، در حالی که `float` فقط حدود ۶ تا ۷ رقم دقت دارد. در زبان C، اعداد اعشاری ثابت مثل `3.14` به‌طور پیش‌فرض از نوع `double` هستند (برای `float` باید پسوند `f` گذاشت: `3.14f`). به همین دلیل، در اکثر برنامه‌ها استفاده از `double` ترجیح داده می‌شود، مگر جایی که حافظه بسیار محدود باشد.

#### نوع منطقی `bool` و نکته‌ی مهم آن

در C مدرن (از C99 به بعد)، برای کار با مقادیر منطقی سرآیند `<stdbool.h>` را اضافه می‌کنیم تا `bool`، `true` و `false` در دسترس باشند:



In [5]:
%%c
#include <stdbool.h>
#include <stdio.h>

int main(void) {
    bool isPassed = true;
    bool hasError = false;
    bool check = 25; // هر مقدار غیرصفر به 1 (true) تبدیل می‌شود!

    printf("%d %d %d\n", isPassed, hasError, check); // 1 0 1
    return 0;
}


--- خروجی برنامه ---
1 0 1




**تله‌ی رایج:** `bool` در C در نهایت به‌صورت عدد `0` (برای `false`) یا `1` (برای `true`) ذخیره و چاپ می‌شود. اگر هر عدد غیرصفری مثل `-5` یا `25` را در متغیر `bool` بریزید، مقدار آن به `1` تبدیل می‌شود.

#### یک عدد چطور در حافظه ذخیره می‌شود؟ (مرور فشرده‌ی مبناها در C)

حافظه مثل ردیفی از خانه‌های کوچک است که هر **بیت (Bit)** آن فقط دو حالت **۰** یا **۱** دارد و هر ۸ بیت کنار هم یک **بایت (Byte)** می‌سازند. داده‌ها در حافظه به شکل الگوی بیت‌ها ذخیره می‌شوند؛ برای مثال، عدد ده‌دهی `13` در یک بایت به صورت `00001101` نگه داشته می‌شود (`8 + 4 + 1 = 13`).

> **نگاهی کوتاه به تاریخ مبناهای عددی:** دستگاه ده‌دهیِ موضعی و کاربرد صفر حاصل تکامل تاریخی ریاضیات هند (از جمله آثار **براهمگوپتا** در قرن هفتم میلادی) بود و چندین قرن بعد **گوتفرید ویلهلم لایبنیتس** در سال ۱۷۰۳ دستگاه دودویی را به‌صورت نظام‌مند برای محاسبات شرح داد.

نکته‌ی کلیدی این است که **مبنا فقط شیوه‌ی نوشتن است، نه تغییر مقدار.** عدد «سیزده» در هر مبنایی که نوشته شود، در حافظه همان الگوی `00001101` را دارد. در برنامه‌نویسی C علاوه بر مبنای ۱۰، با **دودویی (مبنای ۲)**، **هشت‌هشتی / Octal (مبنای ۸)** و **شانزده‌شانزدهی / Hex (مبنای ۱۶)** سر و کار داریم؛ چون $8 = 2^3$ و $16 = 2^4$ هستند و هر رقم هگز دقیقاً نماینده‌ی **۴ بیت** و هر رقم هشت‌هشتی نماینده‌ی **۳ بیت** است:

| ده‌دهی | دودویی (۴ بیتی) | هگز | | ده‌دهی | دودویی (۴ بیتی) | هگز |
|:---:|:---:|:---:|---|:---:|:---:|:---:|
| `0` | `0000` | `0` | | `8` | `1000` | `8` |
| `1` | `0001` | `1` | | `9` | `1001` | `9` |
| `2` | `0010` | `2` | | `10` | `1010` | `A` |
| `3` | `0011` | `3` | | `11` | `1011` | `B` |
| `4` | `0100` | `4` | | `12` | `1100` | `C` |
| `5` | `0101` | `5` | | `13` | `1101` | `D` |
| `6` | `0110` | `6` | | `14` | `1110` | `E` |
| `7` | `0111` | `7` | | `15` | `1111` | `F` |

* **تبدیل سریع دودویی ↔ هگز:** بیت‌ها را از **سمت راست** چهارتاچهارتا جدا کنید و هر دسته را با جدول بالا به یک رقم هگز ببرید (برای هشت‌هشتی، دسته‌های سه‌تایی جدا کنید). مثلاً `1010 1111 0011` می‌شود `0xAF3`. برعکس، هر رقم هگز را با ۴ بیت معادلش باز کنید (`0x3B` می‌شود `0011 1011`).
* **تبدیل مبنای $b$ به ده‌دهی:** هر رقم را از راست در توان‌های مبنا ($b^0, b^1, b^2, \dots$) ضرب و جمع کنید. مثلاً:
  $$\text{2F}_{16} = 2 \times 16^1 + 15 \times 16^0 = 32 + 15 = 47$$
* **تبدیل ده‌دهی به مبنای $b$:** عدد را متوالی بر $b$ تقسیم کنید تا خارج‌قسمت صفر شود، سپس باقیمانده‌ها را از **آخر به اول** بخوانید. مثلاً `13` در تقسیم متوالی بر `2` باقیمانده‌های `1, 0, 1, 1` می‌دهد که از آخر به اول می‌شود `1101`.

##### نوشتن و چاپ مبناها در C (و یک تله‌ی بسیار رایج!)

در زبان C، مبنای عدد ثابت با پیشوند آن مشخص می‌شود:



In [6]:
%%c
#include <stdio.h>

int main(void) {
    int a = 13;       // بدون پیشوند -> ده‌دهی
    int b = 0xD;      // پیشوند 0x یا 0X -> هگزادسیمال (حروف کوچک و بزرگ فرقی ندارند)
    int c = 015;      // پیشوند 0 -> هشت‌هشتی (Octal)
    int d = 0b1101;   // پیشوند 0b -> دودویی (استاندارد C23 و افزونه‌ی رایج GCC/Clang)

    printf("%d %d %d %d\n", a, b, c, d);         // خروجی: 13 13 13 13
    printf("Hex: %x | Upper: %X | With prefix: %#x\n", 255, 255, 255); // ff | FF | 0xff
    printf("Octal: %o | With prefix: %#o\n", 13, 13);                   // 15 | 015
    return 0;
}


--- خروجی برنامه ---
13 13 13 13
Hex: ff | Upper: FF | With prefix: 0xff
Octal: 15 | With prefix: 015




**تله‌ی امتحانی و برنامه‌نویسی:** در زبان C، گذاشتن صفر در ابتدای عدد به معنی **مبنای ۸** است! بنابراین `010` برابر با `8` است نه `10`، و نوشتن `int x = 08;` یا `09;` **خطای کامپایل** می‌دهد (چون ارقام ۸ و ۹ در مبنای هشت وجود ندارند). همچنین `%x` به‌تنهایی پیشوند `0x` را چاپ نمی‌کند و برای چاپ پیشوند باید از `%#x` استفاده کنید.

> **برای کنجکاوها:**
> * **کاربردهای واقعی:** هگز در نمایش آدرس‌های حافظه (`%p`)، کد رنگ‌های وب (`#FF5733` یعنی سه بایت قرمز، سبز و آبی) و تحلیل بیت‌ها به کار می‌رود؛ هشت‌هشتی نیز در مجوزهای فایل یونیکس (مثل `chmod 755`) رایج است.
> * **اعشار در دودویی و Endianness:** کسرهایی مثل `0.1` در مبنای ۲ پایان‌پذیر نیستند و رشته‌ی نامتناهی می‌سازند که علت اصلی خطای دقت در `float` و `double` است. همچنین در بسیاری از پردازنده‌های امروزی (معماری Little-endian)، بایت کم‌ارزش‌تر یک عدد چندبایتی در آدرس کوچک‌تر حافظه قرار می‌گیرد.

#### `long` و `long long` — وقتی `int` کافی نیست

همان‌طور که در بخش «حوزه‌ی مقادیر و سرریز» خواهیم دید، `int` معمولاً فقط تا حدود ۲.۱ میلیارد را پوشش می‌دهد. اگر بخواهیم عددی بزرگ‌تر از این (مثلاً جمعیت جهان یا حاصل‌ضرب دو عدد بزرگ) را نگه داریم، باید از `long` یا `long long` استفاده کنیم:



In [7]:
%%c
#include <stdio.h>

int main(void) {
    long long population = 8000000000LL;
    long long distance = 9460730472580800LL; // پسوند LL برای جلوگیری از سرریز مقدار ثابت

    printf("Population: %lld\n", population);
    printf("Distance (light-year): %lld\n", distance);
    return 0;
}


--- خروجی برنامه ---
Population: 8000000000
Distance (light-year): 9460730472580800




نکات کلیدی:

- برای چاپ و خواندن `long` از `%ld` و برای `long long` از `%lld` استفاده کنید (نه `%d`).
- پسوند `L` یا `LL` در انتهای عدد ثابت به کامپایلر می‌گوید که عدد را از همان ابتدا `long` یا `long long` در نظر بگیرد.
- استاندارد C برای `long long` دست‌کم بازه‌ای معادل ۶۴ بیت را تضمین می‌کند، بنابراین برای اعداد بسیار بزرگ قابل‌حمل‌تر از `long` است.

#### اعداد علامت‌دار و بدون علامت (`signed` / `unsigned`)

نوع‌های صحیح در C می‌توانند علامت‌دار یا بدون علامت باشند:



In [8]:
%%c
signed int temperature = -5;
unsigned int count = 42U;



`unsigned` نمی‌تواند مقدار منفی را نمایش دهد و در عوض تمام بیت‌ها را برای اعداد نامنفی استفاده می‌کند (بیشینه‌ی بازه حدوداً دو برابر می‌شود). ترکیب `signed` و `unsigned` در یک عبارت یا مقایسه‌ی عدد منفی با `unsigned` می‌تواند نتیجه‌های غافلگیرکننده ایجاد کند؛ پس در کد مقدماتی بی‌دلیل از `unsigned` استفاده نکنیم.

> **برای کنجکاوها:** در `<stdint.h>` نوع‌هایی مانند `int32_t` و `uint64_t` وجود دارند که وقتی دقیقاً به تعداد مشخصی بیت نیاز داریم مفیدند.

#### دقت اعشاری و خطای نمایش

`float` و `double` همه‌ی اعداد اعشاری را دقیق نگه نمی‌دارند. مثلاً مقدار `0.1` در نمایش دودویی متناهی نیست و به نزدیک‌ترین مقدار قابل‌نمایش گرد می‌شود:



In [9]:
%%c
#include <stdio.h>

int main(void) {
    double x = 0.1 + 0.2;
    printf("%.17f\n", x); // خروجی نزدیک به 0.30000000000000004
    return 0;
}


--- خروجی برنامه ---
0.30000000000000004




نتیجه‌ی آموزشی مهم: برای مقایسه‌ی اعداد اعشاری، همیشه `x == 0.3` بهترین انتخاب نیست.

> **پنجره‌ی بیشتر:** استاندارد رایج نمایش اعداد اعشاری در سخت‌افزار **IEEE 754** نام دارد.
>
> **نگاهی به یک دانشمند تأثیرگذار — ویلیام کاهان (William Kahan):**
> کاهان، ریاضی‌دان و دانشمند علوم کامپیوتر کانادایی، از چهره‌های اصلی طراحی استاندارد IEEE 754 برای محاسبات ممیز شناور بود که به قابل‌پیش‌بینی‌تر شدن محاسبات اعشاری در رایانه‌ها کمک کرد.

#### `char`، ASCII و Unicode: یک تفاوت مهم

`char` در واقع یک **عدد صحیح کوچک ۱ بایتی** است که کد عددی یک کاراکتر (مثلاً کد ASCII) را نگه می‌دارد؛ به همین دلیل هم می‌توان آن را با `%c` به‌صورت کاراکتر و هم با `%d` به‌صورت کد عددی چاپ کرد:



In [10]:
%%c
char ch = 'A';
printf("Char: %c | ASCII code: %d\n", ch, ch); // Char: A | ASCII code: 65
printf("Next letter: %c\n", ch + 1);            // B


--- خروجی برنامه ---
Char: A | ASCII code: 65
Next letter: B




دانستن سه کد پایه‌ی جدول ASCII در برنامه‌نویسی بسیار پرکاربرد است: `'0'` برابر `48`، `'A'` برابر `65` و `'a'` برابر `97` است (فاصله‌ی حروف بزرگ و کوچک انگلیسی همیشه `32` واحد است).

همچنین توجه کنید که **متن فارسی را نمی‌توان با فرض «هر حرف = یک char» تحلیل کرد**. متن‌های Unicode معمولاً با رمزگذاری‌هایی مثل UTF-8 ذخیره می‌شوند و یک نویسه‌ی فارسی ممکن است ۲ بایت یا بیشتر اشغال کند.

> **نگاهی به چهره‌های اثرگذار — رابرت بمر و جو بکر:**
> **رابرت دبلیو. بمر (Robert W. Bemer)** از چهره‌های مهم در شکل‌گیری و گسترش استاندارد ASCII بود. با گسترش استفاده از زبان‌های گوناگون، **جو بکر (Joe Becker)** و همکارانش پروژه‌ی Unicode را پایه‌گذاری کردند تا نویسه‌های تمام زبان‌های جهان در یک استاندارد واحد پوشش داده شوند.

### 1.3 قواعد نام‌گذاری متغیرها

نام متغیر نمی‌تواند دلخواه باشد؛ باید از قواعد زیر پیروی کند:

- فقط می‌تواند شامل حروف، اعداد و آندرلاین (`_`) باشد.
- نمی‌تواند با عدد شروع شود (`2value` غلط است، `value2` درست است).
- به بزرگی و کوچکی حروف حساس است (`Age` و `age` دو متغیر متفاوت‌اند).
- نمی‌تواند از کلمات کلیدی زبان (مثل `int`, `for`, `return`) استفاده کند.
- بهتر است نامی معنادار و خواناکه انتخاب شود (`studentCount` بهتر از `x` است).

```c
int studentCount = 30;   // خوب
int 2ndYear = 5;         // خطای کامپایل (شروع با عدد)
int student count = 10;  // خطای کامپایل (فاصله مجاز نیست)
```

**سبک‌های رایج نام‌گذاری (Naming Convention):** قواعد بالا فقط می‌گویند چه نامی «مجاز» است؛ اما برنامه‌نویس‌ها برای نام‌های چندکلمه‌ای (مثل «تعداد دانشجو») روی چند *سبک* قرارداد کرده‌اند. خود C سبک خاصی را اجباری نکرده است.

| سبک                      | مثال                | منشأ و جاهایی که استفاده می‌شود                                                                                                                   |
|--------------------------|---------------------|--------------------------------------------------------------------------------------------------------------------------------------------------|
| `camelCase`              | `studentCount`      | ریشه در جامعه‌ی زبان Smalltalk (دهه‌ی ۱۹۷۰)؛ امروز برای متغیر و تابع در Java و JavaScript رایج است.                                                |
| `PascalCase`             | `StudentCount`      | به‌نام زبان Pascal (نیکلاوس ویرت، ۱۹۷۰)؛ برای نام کلاس و نوع در Java و C#، و در Windows API.                                                      |
| `snake_case`             | `student_count`     | سنتی در Unix و C؛ کتابخانه‌ی استاندارد C (مثل `size_t` و `strlen`)، هسته‌ی لینوکس، Python (استاندارد PEP 8) و Rust.                                |
| `UPPER_SNAKE_CASE`       | `MAX_STUDENTS`      | مخصوص ثابت‌ها و ماکروها در C، مثل `INT_MAX` در `<limits.h>`.                                                                                      |
| نت مجارستانی (Hungarian) | `iCount` ، `szName` | ابداع چارلز سیمونی (Charles Simonyi)، برنامه‌نویس Xerox PARC و بعد Microsoft؛ در Windows API (مثل `hWnd`) دیده می‌شود، اما امروز کمتر توصیه می‌شود. |
| `kebab-case`             | `student-count`     | در C مجاز نیست (`-` عملگر منفی است)؛ در CSS و آدرس وب‌سایت‌ها استفاده می‌شود.                                                                       |

> **نگاهی کوتاه به چارلز سیمونی (Charles Simonyi):** سیمونی، مهندس نرم‌افزار مجارستانی-آمریکایی، در Xerox PARC و سپس Microsoft کار کرد و روش نام‌گذاری Hungarian Notation را رواج داد. ایده‌ی آن، افزودن پیشوندهایی به نام متغیر برای نشان دادن نوع یا نقش آن بود. این روش از نظر تاریخی اثرگذار است، اما امروزه در بسیاری از سبک‌راهنماها نام‌های روشن و معنادار بدون پیشوندهای نوع ترجیح داده می‌شوند.



In [11]:
%%c
int student_count = 30;  // snake_case
int studentCount = 30;   // camelCase
#define MAX_STUDENTS 100 // UPPER_SNAKE_CASE (ثابت/ماکرو)



این سبک‌ها «قانون زبان» نیستند، بلکه قرارداد تیم‌ها و راهنمای سبک کدنویسی‌اند (مثل *Linux kernel coding style*، *PEP 8* در Python و راهنمای کدنویسی Google). در درس، هر سبکی را انتخاب کردید، در کل برنامه همان را ادامه دهید.

#### چند ظرافت نام‌گذاری که ارزش دانستن دارند

- نام‌هایی که با دو آندرلاین (`__name`) شروع می‌شوند یا الگوهای خاصی از آندرلاین و حرف بزرگ دارند، برای پیاده‌سازی زبان/کتابخانه رزرو شده‌اند؛ برای کد دانشجویی از آن‌ها دوری کنید.
- نام متغیر **معنادار** باشد، اما لازم نیست تبدیل به یک جمله‌ی طولانی شود.
- سبک‌هایی مثل `camelCase` و `snake_case` هر دو رایج‌اند؛ مهم‌تر از انتخاب یکی، **ثبات** در کل پروژه است.
- مفهوم **Scope** در فصل توابع کامل‌تر بررسی می‌شود؛ فعلاً بدانید دو متغیر هم‌نام در محدوده‌های متفاوت الزاماً یک متغیر نیستند.

### 1.4 حوزه‌ی مقادیر و سرریز (Overflow)

هر نوع داده، به‌خاطر تعداد بیت‌های محدودی که در حافظه اشغال می‌کند، فقط می‌تواند بازه‌ی مشخصی از اعداد را نگه دارد. برای مثال `int` معمولاً بین `-۲٬۱۴۷٬۴۸۳٬۶۴۸` تا `۲٬۱۴۷٬۴۸۳٬۶۴۷` را پوشش می‌دهد.

وقتی نتیجه‌ی یک محاسبه از بازه‌ی نوع داده خارج شود، با **سرریز (Overflow)** روبه‌رو می‌شویم. نکته‌ی علمی مهم این است که در C، رفتار **سرریز عدد صحیح علامت‌دار** (`signed int`) طبق استاندارد **تعریف‌نشده (Undefined Behavior)** است؛ بنابراین نباید روی برگشتن خودکار عدد از انتهای بازه به ابتدای آن حساب کنیم. در مقابل، سرریز انواع `unsigned` طبق حساب پیمانه‌ای تعریف شده است (مثلاً در `unsigned char` با بیشینه‌ی `255`، حاصل `255 + 1` برابر `0` می‌شود).

```c
#include <limits.h>
#include <stdio.h>

int main(void) {
    int x = INT_MAX;       // بزرگ‌ترین مقدار ممکن برای int
    printf("%d\n", x + 1); // وارد محدوده‌ی Undefined Behavior می‌شود!
    return 0;
}
```

#### چطور بازه‌ی واقعی نوع‌ها را از خود C بپرسیم؟

به‌جای حفظ‌کردن عددها، می‌توان از هدر `<limits.h>` (و برای اعشاری‌ها از `<float.h>`) استفاده کرد:



In [12]:
%%c
#include <limits.h>
#include <stdio.h>

int main(void) {
    printf("INT_MIN  = %d\n", INT_MIN);
    printf("INT_MAX  = %d\n", INT_MAX);
    printf("UINT_MAX = %u\n", UINT_MAX);
    return 0;
}


--- خروجی برنامه ---
INT_MIN  = -2147483648
INT_MAX  = 2147483647
UINT_MAX = 4294967295




#### چهار واژه‌ی خطا که از همین حالا باید بشناسیم

| نوع مشکل | نمونه | معمولاً چه زمانی دیده می‌شود؟ |
|---|---|---|
| **Syntax Error** | • جا انداختن `;` یا `"` پایانی<br>• باز یا بسته‌نشدن پرانتز و آکولاد: `(` `{`<br>• نام متغیر با رقم اول: `int 2x;`<br>• جا انداختن ویرگول در `printf("%d" x);` | زمان کامپایل (کد کامپایل نمی‌شود) |
| **Warning** | • متغیر تعریف‌شده ولی استفاده‌نشده<br>• ناهماهنگی format specifier، مثل چاپ `double` با `%d`<br>• خواندن متغیر مقداردهی‌نشده<br>• ریختن عدد بزرگ در نوع کوچک، مثل `char c = 300;` | زمان کامپایل (فایل ساخته می‌شود ولی هشدار دارد) |
| **Runtime Error** | • تقسیم بر صفر صحیح در زمان اجرا (`x / 0`)<br>• باقیمانده‌ی تقسیم بر صفر (`x % 0`)<br>• دسترسی غیرمجاز به حافظه | هنگام اجرای برنامه (برنامه متوقف می‌شود) |
| **Logic Error** | • تقسیم صحیح به‌جای اعشاری: `5 / 2` برابر `2` می‌شود<br>• اولویت اشتباه عملگرها: `a + b / 2` به‌جای `(a + b) / 2`<br>• استفاده از `=` به‌جای `==` | برنامه اجرا می‌شود ولی خروجی غلط است |

> **مفهوم مهم‌تر: Undefined Behavior.** بعضی عملیات در C آن‌قدر خطرناک‌اند که استاندارد نتیجه‌ی مشخصی برایشان تعریف نمی‌کند؛ مثل سرریز `signed int` یا استفاده از متغیر مقداردهی‌نشده. کامپایلر در این موارد هیچ تعهدی ندارد.

### 1.5 ثابت‌ها (Constants)

گاهی می‌خواهیم مقداری در طول برنامه **هرگز تغییر نکند**. برای این کار دو روش رایج داریم:

**روش اول: کلمه‌ی کلیدی `const`**



In [13]:
%%c
const double PI = 3.14159;
// PI = 3.0; // خطای کامپایل: نمی‌توان ثابت را تغییر داد



**روش دوم: دستور پیش‌پردازنده `#define`**



In [14]:
%%c
#define PI 3.14159



**تفاوت اصلی (سؤال رایج امتحانی):**
1. `const` یک متغیر واقعی با **نوع داده‌ی مشخص** است که قوانین Scope را رعایت می‌کند و کامپایلر نوع آن را بررسی می‌کند.
2. `#define` یک دستور پیش‌پردازنده است که **قبل از کامپایل** صرفاً یک جایگزینی متنی کورکورانه انجام می‌دهد، نوع داده ندارد و در انتهای آن `;` گذاشته نمی‌شود (گذاشتن `;` در انتهای `#define` یک باگ کلاسیک است!).

#### Literalها و پسوندها

خودِ عددهایی که مستقیماً در کد می‌نویسیم **literal** نام دارند. پسوندها نوع آن‌ها را برای کامپایلر دقیق می‌کنند:

```c
42       // int
42U      // unsigned int
42L      // long
42LL     // long long
3.14     // double (پیش‌فرض اعداد اعشاری)
3.14f    // float
```

### 1.6 تبدیل نوع (Type Casting)

گاهی لازم است مقدار یک نوع داده را به نوع دیگری تبدیل کنیم.

**تبدیل ضمنی (Implicit Casting):** کامپایلر خودش، بدون این‌که از او بخواهیم، تبدیل را انجام می‌دهد (معمولاً از نوع کوچک‌تر به بزرگ‌تر، مثلاً از `int` به `double`):



In [15]:
%%c
int a = 5;
double b = a; // int به‌طور خودکار به double تبدیل می‌شود (5.0)



**تبدیل صریح (Explicit Casting):** خودمان به‌صراحت نوع مقصد را داخل پرانتز می‌نویسیم:



In [16]:
%%c
double result = 7 / 2;          // نتیجه: 3.0 (چون هر دو عدد int هستند و تقسیم صحیح انجام می‌شود)
double correct = (double)7 / 2; // نتیجه: 3.5 (ابتدا 7 به double تبدیل می‌شود، سپس تقسیم اعشاری رخ می‌دهد)



> **تله‌ی بسیار رایج (مراقب جای پرانتز باشید!):**
> عبارت `(double)(7 / 2)` باز هم نتیجه را **`3.0`** می‌دهد! زیرا پرانتز باعث می‌شود اول تقسیم صحیح `7 / 2` انجام شود (که حاصلش `3` است) و سپس `3` به `3.0` تبدیل شود. برای گرفتن نتیجه اعشاری، حداقل یکی از عملوندها باید **قبل از تقسیم** تبدیل شود: `(double)7 / 2` یا `7.0 / 2`.

#### تبدیل کاهشی و Integer Promotion

تبدیل از نوع بزرگ‌تر به کوچک‌تر باعث از دست رفتن اطلاعات می‌شود:



In [17]:
%%c
double price = 19.99;
int whole = (int)price; // 19؛ بخش اعشاری حذف (Truncate) می‌شود، گرد نمی‌شود!



همچنین انواع کوچک‌تر مثل `char` و `short` در محاسبات ابتدا به‌طور خودکار به `int` ارتقا پیدا می‌کنند (**Integer Promotion**). اگر دو نوع متفاوت در یک عملگر ریاضی کنار هم قرار بگیرند (مثلاً `int` و `double`)، نوع کوچک‌تر به نوع بزرگ‌تر ارتقا می‌یابد.

#### تمرین خطایابی کوتاه: خروجی را قبل از اجرا پیش‌بینی کنید

```c
int a = 7;
int b = 2;
double x = a / b;
double y = (double)a / b;
printf("%.1f %.1f\n", x, y);
```

1. خروجی `x` چیست؟
2. خروجی `y` چیست؟
3. تبدیل نوع در کدام مرحله باید رخ دهد تا بخش اعشاری از بین نرود؟

### تمرین‌های پایانی فصل ۱

- انواع داده: `int`, `float`, `double`, `char`, `bool`
- قواعد نام‌گذاری معتبر
- بازه‌ی مقادیر و خطر Overflow
- ثابت‌ها: `const` در برابر `#define`
- تبدیل نوع: implicit در برابر explicit

#### تمرین ۱ — محاسبه‌ی سن

برنامه‌ای بنویسید که سن کاربر را در یک `int` بگیرد، آن را در ۲ (به‌عنوان `double`) ضرب کند و با تبدیل نوع صریح چاپ کند.

#### تمرین ۲ — رشد باکتری و کشف Overflow

فرض کنید یک جمعیت باکتری از `1` شروع می‌شود و در هر مرحله **دو برابر** می‌شود.

برنامه‌ای بنویسید که یک متغیر از نوع `int` را از `1` شروع کند و در هر مرحله مقدار آن را در `2` ضرب کند. بعد از هر مرحله مقدار را چاپ کنید و ببینید با زیاد شدن مقدار، چه اتفاقی برای عدد می‌افتد.

**هدف تمرین:** می‌خواهیم Overflow را فقط در تئوری نبینید؛ خودتان رشد سریع عدد را اجرا کنید و ببینید وقتی مقدار از ظرفیت نوع داده عبور کند چه مشکلی به وجود می‌آید.

> **راهنمایی:** فعلاً نیازی به حلقه ندارید. می‌توانید چند بار پشت سر هم بنویسید:
>
> ```c
> x *= 2;
> printf("%d\n", x);
> ```
>
> و مرحله‌به‌مرحله خروجی را بررسی کنید.
>
> **نکته:** رفتار Overflow برای `int` علامت‌دار در استاندارد C تضمین‌شده نیست؛ هدف این تمرین مشاهده‌ی مفهوم سرریز در عمل است. برای آزمایش کاملاً قابل‌پیش‌بینی، بعداً می‌توانید همین ایده را با انواع unsigned بررسی کنید.

#### تمرین ۳ — مبدل فارنهایت به سلسیوس

برنامه‌ای بنویسید که یک دما بر حسب فارنهایت بگیرد و آن را به سلسیوس تبدیل کند (`C = (F - 32) * 5 / 9`).

> این یکی از قدیمی‌ترین و معروف‌ترین تمرین‌های آموزش C است؛ نسخه‌ی اصلی آن دقیقاً همین برنامه است که در کتاب کلاسیک ***The C Programming Language*** نوشته‌ی Brian Kernighan و Dennis Ritchie (معروف به کتاب K&R) به‌عنوان یکی از اولین برنامه‌های آموزشی کتاب معرفی شده است.

#### تمرین ۴ — محاسبه‌گر BMI

برنامه‌ای بنویسید که قد (بر حسب متر) و وزن (بر حسب کیلوگرم) را بگیرد و شاخص توده‌ی بدنی را حساب کند (`BMI = weight / (height * height)`).

> این تمرین یک نمونه‌ی کلاسیک برای تمرین متغیرهای اعشاری و ترتیب محاسبه است.

#### تمرین ۵ — شکار خطا: نوع داده‌ی نامناسب

کد زیر را بررسی کنید و بدون اجرا بگویید چه اشکالاتی دارد. سپس آن را اصلاح کنید:

```c
int average = 17.8;
char grade = "A";
float pi = 3.141592653589793;
printf("%d %c %f\n", average, grade, pi);
```

هدف، تشخیص تفاوت **کاراکتر و رشته**، تبدیل ضمنی و محدودیت دقت `float` است.

#### تمرین ۶ — مقایسه‌ی `float` و `double`

عدد `1.0 / 3.0` را یک‌بار در `float` و یک‌بار در `double` ذخیره کنید و هر دو را با تعداد زیادی رقم اعشار چاپ کنید. تفاوت را توضیح دهید.

#### تمرین ۷ — بازه‌ی نوع‌ها با `<limits.h>`

برنامه‌ای بنویسید که `CHAR_BIT`، `INT_MIN`، `INT_MAX` و `UINT_MAX` را چاپ کند. سپس حدس بزنید سیستم شما برای `int` چند بیت استفاده می‌کند.

---

### Retrieval Practice فصل ۱

بدون نگاه‌کردن به متن، به پرسش‌های زیر پاسخ دهید. اگر در پاسخ‌دادن به موردی مکث کردید، همان زیربخش را دوباره مرور کنید.

1. تفاوت Declaration، Initialization و Assignment چیست؟
2. چرا اندازه‌ی `int` را نباید در همه‌ی سیستم‌ها الزاماً ۴ بایت فرض کنیم؟
3. تفاوت نمایش یک عدد در مبناهای ۲، ۸، ۱۰ و ۱۶ چیست؟ آیا با تغییر مبنا، مقدار عدد عوض می‌شود؟
4. تفاوت `signed` و `unsigned` چیست و چرا سرریز عدد صحیح علامت‌دار در C قابل اتکا نیست؟
5. تفاوت `const` و `#define` در چیست؟
6. چه زمانی تبدیل نوع ضمنی رخ می‌دهد و چه زمانی باید از تبدیل صریح استفاده کنیم؟

### 1.7 جمع‌بندی فصل ۱

در این فصل دیدیم که متغیرها نام‌هایی برای دسترسی به داده‌های برنامه‌اند و نوع داده مشخص می‌کند مقدار چگونه تفسیر شود. نمایش داخلی داده‌ها بر پایه‌ی بیت‌هاست؛ مبناهای عددی شیوه‌های متفاوت نوشتن یک مقدارند و اندازه و بازه‌ی انواع داده به پیاده‌سازی وابسته است. همچنین با قواعد نام‌گذاری، ثابت‌ها، سرریز و تبدیل نوع آشنا شدیم.

| مفهوم | نکته‌ی کلیدی |
|---|---|
| Variable | نامی برای دسترسی به یک مقدار در برنامه |
| Data Type | تعیین‌کننده‌ی تفسیر، اندازه و عملیات مجاز روی مقدار |
| Number Bases | شیوه‌های متفاوت نمایش یک مقدار واحد |
| Overflow | عبور مقدار از بازه‌ی قابل نمایش نوع داده |
| Constants | مقادیری که در طول اجرای برنامه نباید تغییر کنند |
| Type Casting | تبدیل مقدار از یک نوع به نوع دیگر |

---

## فصل ۲ : ورودی/خروجی و عملگرهای پایه

### 2.1 چطور با برنامه ارتباط برقرار کنیم؟

تا اینجا دیدیم که متغیرها داده را در برنامه نگه می‌دارند؛ اما یک برنامه‌ی کاربردی نباید فقط با مقادیری کار کند که از قبل در کد نوشته‌ایم. معمولاً لازم است کاربر اطلاعاتی وارد کند، برنامه آن‌ها را پردازش کند و نتیجه را به شکلی خوانا نمایش دهد. از طرف دیگر، برای حل مسئله‌ها باید بتوانیم روی داده‌ها محاسبه انجام دهیم، آن‌ها را با هم مقایسه کنیم یا چند شرط را به هم پیوند بدهیم.

در این فصل ابتدا با ورودی و خروجی استاندارد در C آشنا می‌شویم. سپس عملگرهای مختلف را از محاسبات ساده تا عملیات بیتی بررسی می‌کنیم. در پایان هم یاد می‌گیریم وقتی چند عملگر در یک عبارت حضور دارند، اولویت و جهت وابستگی آن‌ها چگونه بر تفسیر عبارت اثر می‌گذارد.

### 2.2 ورودی و خروجی: `printf` و `scanf`

#### `printf` — چاپ خروجی و قالب‌بندی دقیق

تابع `printf` از **Format Specifier**ها (مشخص‌کننده‌های قالب) استفاده می‌کند تا بداند هر مقدار را با چه نوع و قالبی چاپ کند:

| Specifier | نوع داده در `printf` | نوع داده در `scanf` |
|:---:|---|---|
| `%d` یا `%i` | `int` | `int *` |
| `%u` | `unsigned int` | `unsigned int *` |
| `%ld` / `%lld` | `long` / `long long` | `long *` / `long long *` |
| `%f` | `float` و `double` | `float *` **(فقط `float`!)** |
| `%lf` | `double` (از C99 مجاز است، ولی `%f` رایج‌تر است) | `double *` **(برای `double` اجباری!)** |
| `%c` | `char` | `char *` |
| `%s` | رشته (`char[]`) | `char[]` (بدون `&`) |
| `%x` / `%X` / `%o` | نمایش هگز / اکتال | خواندن هگز / اکتال |
| `%zu` | `size_t` (خروجی `sizeof`) | `size_t *` |
| `%p` | آدرس حافظه (`(void*)`) | — |
| `%%` | چاپ خود علامت `%` | — |

##### کنترل عرض، دقت و تراز در `printf`

با قرار دادن عدد بین `%` و حرف قالب، می‌توان ظاهر خروجی را دقیق تنظیم کرد:



In [18]:
%%c
double gpa = 17.4567;
int id = 42;

printf("%.2f\n", gpa);    // 17.46 (گرد کردن تا ۲ رقم اعشار)
printf("%8.2f\n", gpa);   // "   17.46" (حداقل عرض ۸ کاراکتر، تراز به راست)
printf("%-8.2f|\n", gpa); // "17.46   |" (علامت منفی یعنی تراز به چپ)
printf("%05d\n", id);     // 00042 (پر کردن فضای خالی سمت چپ با صفر)
printf("%.*f\n", 3, gpa); // 17.457 (تعیین تعداد اعشار به‌صورت پویا با آرگومان)


--- خروجی برنامه ---
17.46
   17.46
17.46   |
00042
17.457




چند **Escape Sequence** (دنباله‌ی فرار) پرکاربرد:

| دنباله | معنی | دنباله | معنی |
|:---:|---|:---:|---|
| `\n` | خط جدید (Newline) | `\"` | چاپ `"` داخل رشته |
| `\t` | تب افقی (Tab) | `\\` | چاپ خود کاراکتر `\` |

#### تمام تله‌ها و نکات مهم `printf` و `scanf` (بسیار مهم در آزمون‌ها!)

توابع `printf` و `scanf` تعداد آرگومان‌های متغیر می‌گیرند و کامپایلر همیشه نمی‌تواند اشتباهات داخل رشته قالب را اصلاح کند. موارد زیر مهم‌ترین دام‌های این دو تابع هستند:

1. **تله‌ی کلاسیک `%f` در برابر `%lf` در `scanf`:**
   در `printf`، آرگومان‌های `float` به‌طور خودکار به `double` ارتقا پیدا می‌کنند؛ بنابراین `%f` برای هر دو کار می‌کند. اما در `scanf`، آدرس حافظه ارسال می‌شود و اندازه حیاتی است: برای `float` حتماً `%f` و برای `double` حتماً **`%lf`** بنویسید. استفاده از `%f` برای `double` در `scanf` باعث خراب شدن مقدار متغیر می‌شود.
2. **فراموش کردن `&` در `scanf` (و استثنای `%s`):**
   `scanf` برای تغییر مقدار متغیر به **آدرس** آن نیاز دارد (`scanf("%d", &age);`). فراموش کردن `&` یکی از رایج‌ترین علت‌های Crash کردن برنامه (Runtime Error) است. تنها استثنا رشته‌ها (`%s`) هستند که نام آرایه خودش نشان‌دهنده‌ی آدرس شروع است.
3. **تله‌ی بافر ورودی و `\n` جا مانده قبل از `%c` (دام شماره یک در کدنویسی!):**
   وقتی کاربر عددی را تایپ می‌کند و کلید `Enter` را می‌زند، مثلاً `20\n` وارد بافر ورودی می‌شود. `scanf("%d", &age)` عدد `20` را برمی‌دارد اما کاراکتر `\n` را در بافر باقی می‌گذارد! اگر بلافاصله بنویسید `scanf("%c", &grade);`، تابع بدون منتظر ماندن برای کاربر، همان `\n` باقی‌مانده را می‌خواند.
   **راه‌حل:** یک **فاصله (Space)** قبل از `%c` بگذارید:
   ```c
   scanf("%d", &age);
   scanf(" %c", &grade); // این فاصله باعث می‌شود تمام Enterها و فاصله‌های قبلی نادیده گرفته شوند
   ```
4. **رفتار `scanf` با فاصله‌ها (Whitespace):**
   قالب‌های عددی (`%d`, `%f`, `%lf`) و `%s` به‌طور خودکار تمام فاصله‌ها، تب‌ها و `Enter`های ابتدای ورودی را رد می‌کنند و هنگام رسیدن به اولین فاصله متوقف می‌شوند. بنابراین `scanf("%s", name)` فقط **یک کلمه** می‌خواند و با زدن Space متوقف می‌شود. برای جلوگیری از سرریز آرایه در رشته، همیشه حداکثر طول را مشخص کنید (`scanf("%19s", name);` برای آرایه‌ی ۲۰ تایی).
5. **کاراکترهای اضافی در رشته‌ی قالب `scanf`:**
   اگر بنویسید `scanf("%d,%d", &a, &b);`، کاربر **مجبور** است بین دو عدد دقیقاً ویرگول تایپ کند (مثلاً `10,20`)؛ در غیر این صورت خواندن عدد دوم شکست می‌خورد. همچنین هرگز در انتهای رشته‌ی قالب `scanf` کاراکتر `\n` نگذارید (`scanf("%d\n", &a)` غلط است)، چون باعث می‌شود برنامه پس از زدن Enter همچنان منتظر یک ورودی غیرسفید بماند!
6. **ناهماهنگی تعداد یا نوع آرگومان‌ها با Format Specifierها:**
   اگر در `printf` سه `%d` بنویسید ولی دو متغیر بدهید، یا یک `double` را با `%d` چاپ کنید، خروجی کاملاً بی‌معنی و دچار **Undefined Behavior** می‌شود.
7. **مقدار بازگشتی `printf` و `scanf`:**
   - `scanf` تعداد آیتم‌هایی را که با موفقیت خوانده و تبدیل کرده برمی‌گرداند (مثلاً اگر کاربر به‌جای عدد، حروف وارد کند، خروجی `0` می‌شود).
   - `printf` تعداد کاراکترهایی را که چاپ کرده برمی‌گرداند (مثلاً `printf("Hi\n")` مقدار `3` برمی‌گرداند).

#### مثال کامل: بررسی بافر، آدرس رشته و کاراکتر تکی



In [19]:
%%c Banafsheh\nA\n
#include <stdio.h>

int main(void) {
    char name[50];
    char first_letter;

    printf("Enter your first name: ");
    scanf("%49s", name); // آرایه name خودش آدرس خانه اول است؛ & نمی‌خواهد

    printf("Enter your favorite letter: ");
    scanf(" %c", &first_letter); // فاصله قبل از %c برای رد کردن Enter قبلی ضروری است

    printf("\n--- Result ---\n");
    printf("Name: %s\n", name);
    printf("Address of name (name):     %p\n", (void*)name);
    printf("Address of name (&name[0]): %p\n", (void*)&name[0]);
    printf("Favorite letter: %c (Address: %p)\n", first_letter, (void*)&first_letter);

    return 0;
}



نمونه خروجی ترمینال:

```text
Enter your first name: Banafsheh
Enter your favorite letter: A

--- Result ---
Name: Banafsheh
Address of name (name):     0x7ff7bfe2f820
Address of name (&name[0]): 0x7ff7bfe2f820
Favorite letter: A (Address: 0x7ff7bfe2f853)
```

**نکته‌ی کلیدی:** نام آرایه در بیشتر عبارت‌ها به اشاره‌گری به عنصر اول آرایه تبدیل می‌شود (`array-to-pointer decay`)؛ به همین دلیل `name` و `&name[0]` آدرس یکسانی نشان می‌دهند و `scanf("%s", name)` نیازی به `&` ندارد.

### 2.3 عملگرهای محاسباتی، رابطه‌ای و منطقی

#### عملگرهای محاسباتی (Arithmetic)

| عملگر | معنی | مثال و نکته |
|:---:|---|---|
| `+` | جمع | `5 + 2` $\rightarrow$ `7` |
| `-` | تفریق (و علامت منفی یکانی) | `5 - 2` $\rightarrow$ `3` |
| `*` | ضرب | `5 * 2` $\rightarrow$ `10` |
| `/` | تقسیم | `5 / 2` $\rightarrow$ `2` (تقسیم صحیح: بخش اعشاری دور ریخته می‌شود!) |
| `%` | باقی‌مانده (Modulo) | `5 % 2` $\rightarrow$ `1` (**فقط برای اعداد صحیح!** روی `float` و `double` خطای کامپایل می‌دهد) |

**تله‌ی امتحانی تقسیم و باقی‌مانده:**
* عملگر `%` روی اعداد اعشاری کار نمی‌کند (`5.5 % 2` خطای کامپایل است).
* در C مدرن، علامت حاصل `a % b` همیشه مطابق با علامت مقسوم (`a`) است؛ مثلاً `-7 % 3` برابر `-1` و `7 % -3` برابر `1` می‌شود (بطوریکه همواره `(a / b) * b + (a % b) == a` برقرار باشد).

#### کاربرد جالب: رمزنگاری سزار (Caesar Cipher)

یکی از قدیمی‌ترین روش‌های رمزنگاری در تاریخ (که به ژولیوس سزار نسبت داده می‌شود) این است: هر حرف پیام را چند جایگاه در الفبا جلو ببریم. چون هر `char` در واقع یک عدد (کد ASCII) است، این کار فقط با **جمع عددی** انجام می‌شود:



In [20]:
%%c
#include <stdio.h>

int main(void) {
    char letter = 'A';
    int shift = 3;

    char encrypted = letter + shift; // 'A' (65) + 3 = 'D' (68)
    printf("Encrypted: %c\n", encrypted); // چاپ می‌کند: D

    return 0;
}


--- خروجی برنامه ---
Encrypted: D




برای رمزگشایی، کافی است همین مقدار `shift` را از حرف رمزشده کم کنیم. نکته‌ی مهم: این نسخه‌ی ساده فقط برای یک حرف تکی و بدون عبور از انتهای الفبا (`Z` به `A`) کار می‌کند؛ نسخه‌ی کامل‌تر که همه‌ی این حالت‌ها را هم پوشش بدهد، بعد از یادگیری `if` و حلقه‌ها قابل تکمیل است.

#### عملگرهای رابطه‌ای (Relational)

نتیجه‌ی این عملگرها همیشه `0` (غلط) یا `1` (درست) است:

| عملگر | معنی |
|:---:|---|
| `==` | برابر است با |
| `!=` | برابر نیست با |
| `<` | کوچک‌تر |
| `>` | بزرگ‌تر |
| `<=` | کوچک‌تر یا مساوی |
| `>=` | بزرگ‌تر یا مساوی |

> **تله‌ی بزرگ امتحانی (زنجیره کردن مقایسه‌ها):**
> در ریاضی می‌نویسیم `1 < x < 5`، اما در C این عبارت **خطای منطقی** است! زیرا `<` چپ‌به‌راست اجرا می‌شود: ابتدا `1 < x` به `0` یا `1` تبدیل می‌شود و سپس `0 < 5` یا `1 < 5` بررسی می‌شود که **همیشه درست (`1`) است** (حتی اگر `x = 100` باشد!). در C باید بنویسید: `(1 < x) && (x < 5)`.
> همچنین هرگز `=` (انتساب) را با `==` (مقایسه) اشتباه نگیرید.

#### عملگرهای منطقی (Logical)

برای ترکیب چند شرط با هم استفاده می‌شوند:

> **نگاهی به یک دانشمند تأثیرگذار — جورج بول (George Boole):**
> جورج بول، ریاضی‌دان انگلیسی، در قرن نوزدهم جبر بولی را صورت‌بندی کرد؛ دستگاهی ریاضی که در آن گزاره‌ها و مقدارهای درست و نادرست با عملگرهایی مانند AND، OR و NOT ترکیب می‌شوند. بعدها **کلود شنون (Claude Shannon)** نشان داد که می‌توان از همین جبر برای تحلیل مدارهای کلیدزنی و دیجیتال استفاده کرد. به همین دلیل، منطق بولی هم در برنامه‌نویسی و هم در طراحی سخت‌افزار اهمیت بنیادی دارد.

| عملگر | معنی | مثال |
|:---:|---|---|
| `&&` | و (AND) — هر دو باید درست باشند | `(age >= 18) && (hasID)` |
| `\|\|` | یا (OR) — کافی است یکی درست باشد | `(isAdmin) \|\| (isOwner)` |
| `!` | نقیض (NOT) — درست را غلط و غلط را درست می‌کند | `!isValid` |

#### جدول ارزش منطقی در C

در C هر مقدار صحیح **صفر** (`0`) به عنوان `false` و هر مقدار **غیرصفر** (مثلاً `1`، `42` یا حتی `-5`) به عنوان `true` در نظر گرفته می‌شود. خروجی عملگرهای رابطه‌ای و منطقی همیشه `0` یا `1` است:

| A | B | `!A` | `A && B` | `A \|\| B` |
|:---:|:---:|:---:|:---:|:---:|
| ۰ | ۰ | ۱ | ۰ | ۰ |
| ۰ | غیرصفر | ۱ | ۰ | ۱ |
| غیرصفر | ۰ | ۰ | ۰ | ۱ |
| غیرصفر | غیرصفر | ۰ | ۱ | ۱ |

#### ارزیابی کوتاه‌مدار (Short-Circuit Evaluation — بسیار مهم در آزمون!)

در عملگرهای `&&` و `||`، زبان C همیشه از **چپ به راست** ارزیابی می‌کند و به‌محض این‌که نتیجه‌ی نهایی قطعی شود، سمت راست را **اصلاً اجرا نمی‌کند**:

* در `A && B`: اگر `A` غلط (`0`) باشد، نتیجه قطعاً `0` است؛ پس `B` اجرا نمی‌شود.
* در `A || B`: اگر `A` درست (غیرصفر) باشد، نتیجه قطعاً `1` است؛ پس `B` اجرا نمی‌شود.



In [21]:
%%c
int a = 0, b = 5;
int res1 = (a != 0) && (10 / a > 2); // امن است! چون a != 0 غلط است، تقسیم بر صفر اصلاً اجرا نمی‌شود.
int res2 = (b == 5) || (++a > 0);    // چون سمت چپ درست است، ++a اجرا نمی‌شود و a همچنان 0 می‌ماند!



#### افزایش و کاهش: `++` و `--`

عملگرهای `++` و `--` مقدار متغیر را یک واحد تغییر می‌دهند (فقط روی متغیرها قابل استفاده‌اند، نه روی عدد ثابت مثل `5++`). تفاوت حالت **پیشوندی (Prefix)** و **پسوندی (Postfix)** زمانی ظاهر می‌شود که داخل یک عبارت بزرگ‌تر استفاده شوند:



In [22]:
%%c
int x = 5;
int a = x++; // Postfix: ابتدا مقدار فعلی x (یعنی 5) در a ریخته می‌شود، سپس x می‌شود 6
int b = ++x; // Prefix: ابتدا x یک واحد زیاد می‌شود (می‌شود 7)، سپس در b ریخته می‌شود



برای خوانایی و جلوگیری از خطا، در کدهای واقعی تا حد امکان `++` و `--` را در یک دستور مجزا بنویسید.

### 2.4 عملگرهای انتسابی

علاوه بر `=` ساده، C عملگرهای انتسابی ترکیبی را برای «محاسبه همراه با مقداردهی مجدد» دارد:

| عملگر | معادل | مثال |
|:---:|---|---|
| `+=` | `x = x + y` | `x += 5;` |
| `-=` | `x = x - y` | `x -= 5;` |
| `*=` | `x = x * y` | `x *= 5;` |
| `/=` | `x = x / y` | `x /= 5;` |
| `%=` | `x = x % y` | `x %= 5;` |

**نکته‌ی ظریف امتحانی:** در انتساب ترکیبی، سمت راست ابتدا به‌طور کامل ارزیابی می‌شود (انگار داخل پرانتز است)؛ بنابراین `x *= 2 + 3` معادل `x = x * (2 + 3)` است، **نه** `x = x * 2 + 3`!

### 2.5 عملگرهای بیتی (Bitwise Operators)

عملگرهای بیتی مستقیماً روی تک‌تک بیت‌های نمایش دودویی اعداد صحیح کار می‌کنند.

> **پیوند تاریخی:** کلود شنون در پایان‌نامه‌ی کارشناسی ارشد خود در سال ۱۹۳۷ نشان داد که جبر بولی جورج بول می‌تواند رفتار مدارهای رله‌ای و کلیدزنی را توصیف کند. عملگرهای بیتی در C به ما اجازه می‌دهند در سطح برنامه با همین الگوهای صفر و یک کار کنیم.

| عملگر | نام | ایده‌ی ساده |
|:---:|---|---|
| `&` | AND بیتی | فقط بیت‌هایی که در هر دو عدد `1` هستند، `1` می‌مانند |
| `\|` | OR بیتی | اگر حداقل یکی از دو بیت `1` باشد، خروجی `1` است |
| `^` | XOR بیتی | اگر دو بیت متفاوت باشند، خروجی `1` است |
| `~` | NOT بیتی | هر بیت را معکوس می‌کند (`0` را `1` و `1` را `0`) |
| `<<` | شیفت چپ | بیت‌ها را به چپ می‌برد (معادل ضرب در توان‌های ۲ برای اعداد نامنفی) |
| `>>` | شیفت راست | بیت‌ها را به راست می‌برد (معادل تقسیم صحیح بر توان‌های ۲ برای اعداد نامنفی) |

**تفاوت حیاتی (سؤال قطعی امتحان):** عملگرهای بیتی (`&`, `|`, `~`) را با عملگرهای منطقی (`&&`, `||`, `!`) اشتباه نگیرید!
برای مثال، اگر `a = 2` (`0010`) و `b = 4` (`0100`) باشند:
* `a && b` برابر **`1`** است (چون هر دو غیرصفرند).
* `a & b` برابر **`0`** است (چون هیچ بیتی در جایگاه مشترک هر دو `1` نیست).

#### عمل روی بیت‌ها؛ با چشم ببینیم

برای سادگی، مثال زیر را در قالب ۴ بیت ببینید:

| عملگر | ورودی اول (`5`) | ورودی دوم (`3`) | نتیجه دودویی | نتیجه ده‌دهی |
|:---:|:---:|:---:|:---:|:---:|
| AND (`&`) | `0101` | `0011` | `0001` | `1` |
| OR (`\|`) | `0101` | `0011` | `0111` | `7` |
| XOR (`^`) | `0101` | `0011` | `0110` | `6` |
| NOT (`~`) | `0101` | — | `1010` (در نمایش ۴ بیتی) | — |



In [23]:
%%c
#include <stdio.h>

int main(void) {
    unsigned int a = 5; // 0101
    unsigned int b = 3; // 0011

    printf("%u\n", a & b);  // 1  -> 0001
    printf("%u\n", a | b);  // 7  -> 0111
    printf("%u\n", a ^ b);  // 6  -> 0110
    printf("%u\n", a << 1); // 10 -> 1010 (ضرب در ۲)
    printf("%u\n", a >> 1); // 2  -> 0010 (تقسیم صحیح بر ۲)
    return 0;
}


--- خروجی برنامه ---
1
7
6
10
2




#### سه کاربرد کلاسیک عملگرهای بیتی

1. **تشخیص زوج یا فرد بودن:** کم‌ارزش‌ترین بیت هر عدد فرد `1` و هر عدد زوج `0` است؛ پس `n & 1` برای اعداد فرد `1` و برای اعداد زوج `0` می‌دهد.
2. **جابه‌جایی دو متغیر بدون متغیر کمکی با XOR:** با سه دستور `a ^= b; b ^= a; a ^= b;` مقدار دو متغیر صحیح جابه‌جا می‌شود (زیرا $x \oplus x = 0$ و $x \oplus 0 = x$).
3. **بیت‌ماسک (Bitmask):** استفاده از هر بیت یک متغیر به‌عنوان یک پرچم روشن/خاموش (مثل سطح دسترسی خواندن، نوشتن و اجرا).

> **هشدار علمی:** قوانین شیفت و عملگر `~` روی اعداد علامت‌دار (`signed`) به دلیل نمایش مکمل دو و قوانین استاندارد جزئیات ظریفی دارند؛ برای کارهای بیتی، همیشه استفاده از انواع **`unsigned`** امن‌تر و قابل‌پیش‌بینی‌تر است.

### 2.6 اولویت (Precedence) و جهت وابستگی (Associativity) عملگرها

وقتی در یک عبارت چند عملگر کنار هم قرار می‌گیرند، کامپایلر بر اساس دو قانون آن را گروه‌بندی می‌کند:

1. **اولویت (Precedence):** تعیین می‌کند کدام عملگر زودتر به عملوندهایش متصل شود (مثلاً `*` اولویت بالاتری از `+` دارد؛ پس `2 + 3 * 4` یعنی `2 + (3 * 4)`).
2. **جهت وابستگی (Associativity):** وقتی دو عملگر **اولویت یکسان** دارند، تعیین می‌کند عبارت از چپ به راست گروه‌بندی شود یا از راست به چپ:
   * **چپ به راست (Left-to-Right):** بیشتر عملگرها مثل `* / %` و `+ -` و `< >` و `&&` چپ‌به‌راست هستند. مثلاً `20 / 5 * 2` چون `/` و `*` هم‌اولویت و چپ‌به‌راست‌اند، به‌صورت `(20 / 5) * 2 = 8` محاسبه می‌شود (نه `20 / 10 = 2`).
   * **راست به چپ (Right-to-Left):** عملگرهای یکانی (`!`, `~`, `(type)` cast، `++` پیشوندی) و **عملگرهای انتسابی** (`=`, `+=`, `-=`, ...) راست‌به‌چپ هستند. به همین دلیل دستور زنجیره‌ای زیر معتبر است:
     ```c
     int a, b, c;
     a = b = c = 10; // به‌صورت a = (b = (c = 10)) تفسیر می‌شود؛ همگی 10 می‌شوند
     ```

#### جدول فشرده‌ی اولویت عملگرهای مهم (از بالا به پایین)

| اولویت | دسته عملگرها | نمادها | جهت وابستگی |
|:---:|---|---|:---:|
| ۱ (بالاترین) | پرانتز و پسوندی | `()` ، `++` و `--` پسوندی | چپ به راست |
| ۲ | عملگرهای یکانی | `+` و `-` علامت، `!` ، `~` ، `(type)` ، `sizeof` ، `++` و `--` پیشوندی | **راست به چپ** |
| ۳ | ضربی | `*` ، `/` ، `%` | چپ به راست |
| ۴ | جمعی | `+` ، `-` | چپ به راست |
| ۵ | شیفت بیتی | `<<` ، `>>` | چپ به راست |
| ۶ | رابطه‌ای (مقایسه‌ی ترتیب) | `<` ، `<=` ، `>` ، `>=` | چپ به راست |
| ۷ | تساوی | `==` ، `!=` | چپ به راست |
| ۸ | AND بیتی | `&` | چپ به راست |
| ۹ | XOR بیتی | `^` | چپ به راست |
| ۱۰ | OR بیتی | `\|` | چپ به راست |
| ۱۱ | AND منطقی | `&&` | چپ به راست |
| ۱۲ | OR منطقی | `\|\|` | چپ به راست |
| ۱۳ (پایین‌ترین) | عملگرهای انتسابی | `=` ، `+=` ، `-=` ، `*=` ، `/=` ، `%=` | **راست به چپ** |

**دو نکته‌ی طلایی برای کدنویسی و امتحان:**
* اولویت عملگرهای بیتی (`&`, `^`, `|`) **پایین‌تر** از عملگرهای مقایسه‌ای (`==`, `!=`, `<`, `>`) است! بنابراین عبارت `flags & 1 == 1` به‌صورت `flags & (1 == 1)` تفسیر می‌شود که یک باگ معروف است؛ همیشه بنویسید `(flags & 1) == 1`.
* **اولویت و جهت وابستگی** فقط جای پرانتزها را مشخص می‌کنند، نه ترتیب ارزیابی در زمان اجرا را! عبارت‌هایی مثل `i = i++ + ++i;` یا `printf("%d %d", i, i++);` دارای **Undefined Behavior** هستند و در C هرگز نباید یک متغیر را در یک عبارت هم تغییر داد و هم دوباره خواند.

#### تمرین تشخیص اولویت

پیش از اجرای برنامه، مقدار نهایی را پیش‌بینی کنید؛ سپس عبارت را با پرانتزگذاریِ روشن‌تر بازنویسی کنید:

```c
int x = 2 + 3 * 4 > 10 && 5 % 2;
```

### تمرین‌های پایانی فصل ۲

- `printf` برای خروجی، `scanf` برای ورودی (و اهمیت `&`)
- عملگرهای محاسباتی، رابطه‌ای، منطقی
- عملگرهای انتسابی کوتاه (`+=`, `-=`, ...)

#### تمرین ۱ — بررسی دو عدد

برنامه‌ای بنویسید که دو عدد از کاربر بگیرد و بررسی کند آیا هر دو مثبت هستند یا نه. در صورت مثبت بودن هر دو عدد، مجموع و باقی‌مانده‌ی تقسیم آن‌ها را چاپ کنید.

> **توجه:** در این تمرین **نباید از `if` و `else` استفاده کنید**. هدف این است که با مفاهیم و عملگرهایی که تا اینجا یاد گرفته‌اید، مسئله را حل کنید.

#### تمرین ۲ — سیستم بررسی PIN

یک برنامه‌ی ساده برای ورود به یک سیستم بنویسید.

یک PIN ثابت مثل `1234` با استفاده از `const` تعریف کنید. سپس PIN را **یک بار** از کاربر بگیرید و بررسی کنید که آیا درست است یا نه. در پایان، نتیجه‌ی مناسب را چاپ کنید؛ مثلاً «ورود موفق» یا «PIN اشتباه است».

> **توجه:** در این تمرین **نباید از `if`، `else` یا حلقه استفاده کنید**. مسئله را فقط با مفاهیم و عملگرهایی که تا این بخش یاد گرفته‌اید حل کنید.

#### تمرین ۳ — دستگاه خودپرداز (ATM)

یک مبلغ پول به ریال (مثلاً `870000` ریال) را از کاربر بگیرید و حساب کنید به چند اسکناس `100000` ریالی و چند اسکناس `10000` ریالی نیاز است.

**هدف تمرین:** استفاده‌ی عملی از عملگرهای تقسیم (`/`)، باقی‌مانده (`%`) و عملگرهای انتسابی ترکیبی مثل `/=` و `%=`.

> **راهنمایی:** ابتدا تعداد اسکناس‌های `100000` ریالی را با تقسیم به دست آورید. سپس با استفاده از `%=` مقدار باقی‌مانده‌ی مبلغ را محاسبه کنید و از روی آن تعداد اسکناس‌های `10000` ریالی را به دست آورید.

#### تمرین ۴ — تبدیل ثانیه به ساعت، دقیقه و ثانیه

عددی به‌عنوان کل ثانیه‌ها بگیرید (مثلاً `3725`) و آن را به قالب ساعت:دقیقه:ثانیه چاپ کنید. فقط با `/` و `%` قابل حل است.

#### تمرین ۵ — تعویض مقدار دو متغیر بدون متغیر کمکی

دو عدد از کاربر بگیرید و مقدار آن‌ها را بدون استفاده از یک متغیر سوم (کمکی) با هم جابه‌جا کنید و نتیجه را چاپ کنید.

#### تمرین ۶ — بررسی سال کبیسه

یک سال **میلادی** را از کاربر بگیرید و بررسی کنید که آیا کبیسه است یا نه. قاعده‌ی سال کبیسه در تقویم میلادی به این صورت است:

| شرط | نتیجه |
|---|---|
| بر ۴ بخش‌پذیر **نباشد** | کبیسه نیست |
| بر ۴ بخش‌پذیر باشد، ولی بر ۱۰۰ نه | کبیسه است |
| بر ۱۰۰ بخش‌پذیر باشد، ولی بر ۴۰۰ نه (مثل ۱۹۰۰) | کبیسه نیست |
| بر ۴۰۰ بخش‌پذیر باشد (مثل ۲۰۰۰) | کبیسه است |

نتیجه را فقط با عملگرهای `%`، `&&`، `||` و `!` (بدون `if`) به‌صورت ۰ یا ۱ چاپ کنید.

#### تمرین ۷ — کالبدشکافی خطاهای ورودی/خروجی

هر خط را مستقل بررسی کنید و مشخص کنید مشکل از نوع syntax، format mismatch، منطق یا محدودیت ورودی است:

```c
int x;
double y;
char c;

scanf("%f", &y);
printf("%d\n", y);
scanf("%d", x);
scanf("%c", &c);
```

سپس نسخه‌ی صحیح را بنویسید و توضیح دهید چرا اصلاح شده است.

#### تمرین ۸ — Debug by Warning

کد زیر را با `-Wall -Wextra` کامپایل کنید و هشدارها را بخوانید:

```c
#include <stdio.h>

int main(void) {
    int count = 10;
    double ratio = 3.7;
    printf("count=%f ratio=%d\n", count, ratio);
    return 0;
}
```

بدون اجرای برنامه، از روی warningها مشکل را پیدا و اصلاح کنید.

#### تمرین ۹ — Decode یک بیت‌ماسک ساده

سه قابلیت را با سه بیت در نظر بگیرید: خواندن = بیت 0، نوشتن = بیت 1، اجرا = بیت 2. اگر مقدار مجوز `5` باشد، با `&` مشخص کنید کدام قابلیت‌ها روشن‌اند. این تمرین مقدمه‌ای کوچک برای درک مجوزهای سطح پایین و پرچم‌هاست.

#### تمرین ۱۰ — سؤال مفهومی کوتاه

برای هر عبارت فقط بگویید «ایمن و واضح»، «قابل اجرا ولی گمراه‌کننده»، یا «دارای رفتار تعریف‌نشده/نامعتبر» و دلیل بیاورید:

```c
5 / 2
5.0 / 2
(double)(5 / 2)
(double)5 / 2
INT_MAX + 1
```

---

### Retrieval Practice فصل ۲

1. تفاوت `printf` و `scanf` چیست و چرا در بیشتر فراخوانی‌های `scanf` از `&` استفاده می‌کنیم؟
2. تفاوت `%f` و `%lf` در `printf` و `scanf` چیست؟
3. چرا `5 / 2` و `5.0 / 2` نتیجه‌ی یکسانی ندارند؟
4. عملگرهای `&&` و `||` چه تفاوتی با `&` و `|` دارند؟
5. ارزیابی کوتاه‌مدار چه زمانی از اجرای بخش دوم یک عبارت جلوگیری می‌کند؟
6. عملگرهای بیتی چه تفاوتی با عملگرهای منطقی دارند؟
7. تفاوت Precedence و Associativity چیست؟

### 2.7 جمع‌بندی فصل ۲

در این فصل، راه ارتباط برنامه با کاربر را با `printf` و `scanf` بررسی کردیم و سپس دیدیم عملگرها چگونه محاسبه، مقایسه و ترکیب شرط‌ها را ممکن می‌کنند. عملگرهای بیتی نیز امکان کار مستقیم با نمایش دودویی داده را فراهم می‌کنند. در نهایت، برای خواندن درست عبارت‌های ترکیبی باید هم اولویت عملگرها و هم جهت وابستگی آن‌ها را در نظر بگیریم.

| گروه عملگر | کاربرد اصلی |
|---|---|
| Arithmetic | انجام محاسبات عددی |
| Relational | مقایسه‌ی دو مقدار |
| Logical | ترکیب یا معکوس‌کردن شرط‌ها |
| Assignment | انتساب و به‌روزرسانی مقدار متغیر |
| Bitwise | انجام عملیات روی بیت‌های مقدار |
| Precedence / Associativity | تعیین نحوه‌ی گروه‌بندی عبارت‌ها |

---

## جمع‌بندی بخش دوم

### نقشه مفهومی

```text
                    Data
                      │
                 Variables
                      │
                 Data Types
                      │
       ┌──────────────┼──────────────┐
       ▼              ▼              ▼
 Representation    Range /        Constants /
 in Memory        Overflow       Type Casting
                      │
                      ▼
                Input / Output
                printf / scanf
                      │
                      ▼
                  Operators
       ┌────────┬────────┬────────┬────────┐
       ▼        ▼        ▼        ▼        ▼
 Arithmetic Relational Logical Assignment Bitwise
                      │
                      ▼
             Precedence / Associativity
```

### جدول مرور سریع

| موضوع | نکته‌ی کلیدی |
|---|---|
| متغیر و نوع داده | نوع داده تفسیر و بازه‌ی مقدار را مشخص می‌کند |
| نمایش در حافظه | داده‌ها در نهایت با الگوهای بیت نمایش داده می‌شوند |
| مبناهای عددی | دودویی، هشت‌هشتی، ده‌دهی و هگز نمایش‌های متفاوت یک مقدارند |
| Overflow | رفتار سرریز به نوع داده و قواعد استاندارد C وابسته است |
| ثابت و تبدیل نوع | `const`، `#define` و casting کاربردهای متفاوتی دارند |
| ورودی و خروجی | `printf` برای نمایش و `scanf` برای دریافت داده است |
| عملگرها | هر گروه عملگر نوع مشخصی از پردازش را انجام می‌دهد |
| اولویت و جهت وابستگی | تعیین می‌کنند عبارت چگونه گروه‌بندی و تفسیر شود |

### Retrieval Practice جامع بخش دوم

1. اگر بخواهید نمره‌ی یک دانشجو را با دو رقم اعشار چاپ کنید، چه نوع داده و چه format specifierای انتخاب می‌کنید؟
2. چرا ذخیره‌کردن یک مقدار در `double` لزوماً باعث نمی‌شود محاسبه‌ی قبلیِ انجام‌شده با دو `int` اعشاری شود؟
3. اگر یک مقدار در مبنای ۱۶ نوشته شده باشد، چگونه می‌توان آن را بدون تغییر مقدار به نمایش دودویی تبدیل کرد؟
4. چرا در `scanf` باید نوع متغیر مقصد با format specifier هماهنگ باشد؟
5. چه تفاوتی میان مقایسه‌ی منطقی و عملیات بیتی وجود دارد؟
6. در چه شرایطی استفاده از پرانتز خوانایی یک عبارت را بهتر می‌کند، حتی اگر از نظر اولویت عملگرها ضروری نباشد؟

### تمرین نهایی بخش دوم

برنامه‌ای بنویسید که یک عدد صحیح از کاربر دریافت کند. سپس:

1. با استفاده از عملگر بیتی `&` مشخص کند عدد زوج است یا فرد.
2. با استفاده از عملگر شیفت چپ `<<` مقدار آن را دو برابر کند.
3. مقدار اولیه و مقدار نهایی را با قالب مناسب چاپ کند.

در پایان، توضیح دهید چرا آزمون زوج یا فرد بودن با `& 1` کار می‌کند و شیفت چپ در این مثال چه ارتباطی با ضرب در ۲ دارد.

---

### منابع و مسیر مطالعه‌ی بیشتر

- Brian W. Kernighan & Dennis M. Ritchie, *The C Programming Language (K&R)* — مرجع کلاسیک زبان C.
- CS50x, Week 1: C — نمونه‌ای از شیوه‌ی آموزش متغیرها، نوع‌ها، عملگرها و کامپایل در یک درس مقدماتی علوم کامپیوتر.
- cppreference: C language — مرجع فنی برای نوع‌ها، تبدیل‌ها، عملگرها و کتابخانه‌ی استاندارد.
- ISO C / compiler documentation — برای جزئیات دقیق رفتارهای تعریف‌شده، پیاده‌سازی‌وابسته و Undefined Behavior.
- CSES Problem Set (cses.fi) — مجموعه‌ی سؤال‌های برنامه‌نویسی و الگوریتمی برای تمرین بیشتر؛ در انتخاب برخی تمرین‌های این فایل، با اعمال تغییراتی، از آن کمک گرفته شده است.
- ابزارهای هوش مصنوعی مثل Claude، ChatGPT و Gemini — برای پرسیدن توضیح بیشتر، مثال و رفع ابهام؛ پاسخ‌هایشان را با منابع مرجع بالا و با اجرای واقعی کد بررسی کنید.

> **پیشنهاد به دانشجو:** لازم نیست در شروع همه‌ی این منابع را خط‌به‌خط بخوانید. مهم است بدانید برای پرسش‌های دقیق‌تر، منابع مرجع وجود دارند و بتوانید در زمان نیاز سراغشان بروید.

---

## تیم تهیه و تدوین

### اعضای تیم و تقسیم کار

| **عضو تیم** | **نقش و مشارکت** |
|:---:|---|
| **ملیکا محمدبیگی** | تهیه‌ی پروتوتایپ و نسخه‌ی اولیه‌ی محتوا، شامل ساختار کلی مباحث و تمرین‌های اولیه؛ همچنین طراحی اولیه‌ی فایل تمرین‌ها، و در ادامه ساخت دو فرمت دیگر از همین فایل تمرین و بررسی آن‌ها برای اطمینان از یکسان بودن با فایل HTML اصلی |
| **درسا عناصری** | بازبینی علمی و ویرایشی، توسعه و بهینه‌سازی ساختار آموزشی، افزودن/حذف و بازطراحی تمرین‌ها، افزودن نکات مفهومی و خطاهای رایج، و بررسی ایده‌های آموزشی مورد استفاده در منابع و دوره‌های دانشگاهی معتبر؛ همچنین بررسی علمی فایل تمرین‌ها، سنجش سختی سؤال‌ها و ارتباط آن‌ها با یکدیگر، یکسان‌سازی و مرتب‌سازی فایل تمرین‌ها، اضافه و کم کردن چندین سؤال، و افزودن تذکرات و نکاتی برای استاد تدریس‌کننده در برخی سؤال‌ها |
| **زینب قاراچشمه** |  |

### فرایند تولید و بازبینی

محتوا از یک نسخه‌ی اولیه آغاز شد و سپس با تمرکز بر **صحت علمی، پیوستگی پیش‌نیاز/پس‌نیاز، تناسب با دانشجوی ترم اول، تنوع مثال‌ها، خطاهای رایج و تمرین‌های تشخیصی** توسعه یافت. در نسخه‌ی حاضر محدودیت تعداد جلسه حذف شده تا مدرس بتواند بر اساس سطح کلاس و زمان واقعی، بین «هسته‌ی اصلی» و «مطالب تکمیلی» انتخاب کند.

## شفافیت درباره‌ی استفاده از هوش مصنوعی

برای شفافیت با خوانندگان این فایل، لازم است بگوییم که در تولید این مجموعه از **ابزارهای هوش مصنوعی (AI)** بهره گرفته‌ایم. این کمک در سه بخش بود:

**ایده‌گیری**  
گرفتن پیشنهاد برای ساختار مباحث، مثال‌ها و شیوه‌ی توضیح مفاهیم.

**تولید محتوا**  
کمک در نگارش پیش‌نویس بخش‌هایی از متن و مثال‌ها.

**بیشترین کمک — تولید خود کدها و فرمت‌ها**  
تولید کدهای HTML و فرمت‌های Markdown و Jupyter Notebook، که بیشترین سهم کمک هوش مصنوعی در همین بخش بود.

**با این حال، محتوای نهایی با نظارت انسانی خودِ ما، یعنی اعضای تیم تهیه و تدوین، بازبینی و تکمیل شده است.**
